# Who responds to what: local and incoming garimpo workers

Lead 2's model has two groups (`output/research/mining_economics_leads.md`, E15):
- outside crews, who respond to the returns to a site (the gold price received,
  enforcement);
- local residents, who turn to mining when their traditional economy erodes (drought,
  mining upstream that degrades the river).

This notebook is prediction 5's first test. SIVEP splits garimpo cases by infection
municipality into local workers (living there) and incoming ones (living elsewhere in
Brazil).

- **Sample.** Legal Amazon municipalities with a garimpo case in 2004–2024.
- **Outcomes.**
  - log(1 + local) and log(1 + incoming) garimpo cases;
  - their difference (local minus incoming), whose coefficient is the difference of the
    two responses with its SE. A shock to transmission that scales both groups alike
    drops out of it.
  - Non-garimpo malaria, for comparison.
- **Residents' shocks.**
  - *Drought:* the deficit of annual precipitation (ERA5-Land, the municipality's own
    catchment), standardised within the municipality over 2000–2024, this year and the
    year before. A variant uses precipitation upstream.
  - *Upstream garimpo:* the C11 garimpo share of the river reaches within 200 km (or
    50 km) upstream, in pp, with the municipality's own garimpo share as a control,
    2004–2019.
- **Crews' shocks.** The log BRL gold price × never-exploited deposits (E10), and IBAMA
  presence × post-2019 and × post-2023 on 2015–2024 (E12).

Municipality and year FE (state × year FE as a check), unweighted, Otto-3 clusters and
the wild-cluster bootstrap.

Code: `src/analysis/garimpo_groups.py`. Tables go to `output/analysis/garimpo/groups/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import warnings

import numpy as np
import pandas as pd

from src.analysis import garimpo_groups as P

warnings.filterwarnings("ignore", message=r".*singleton fixed effect")
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda value: f"{value:.4g}")

OUT = PROJECT_ROOT / "output" / "analysis" / "garimpo" / "groups"
OUT.mkdir(parents=True, exist_ok=True)

frame = P.build_frame()
sample = frame[frame[P.SAMPLE] > 0]
print(sample["mun_id"].nunique(), "garimpo municipalities of", frame["mun_id"].nunique())
display(sample[sample["year"].between(*P.YEARS)].groupby("year")[
    ["inf_garimpo", "inf_garimpo_local", "inf_garimpo_incoming"]].sum().astype(int).T)

437 garimpo municipalities of 772


year,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
inf_garimpo,16304,15096,12520,12725,11388,13393,14290,10126,11110,9800,5350,3933,4563,4356,4417,8663,15126,19829,22284,18193,14609
inf_garimpo_local,9724,9937,8830,9171,7677,9592,10660,7361,8146,7433,3764,2669,2936,2505,2372,4531,7833,8695,6714,8720,10315
inf_garimpo_incoming,6577,5159,3689,3552,3709,3799,3628,2759,2963,2362,1585,1262,1627,1851,2045,4129,7292,11131,15569,9469,4293


**Reading the results.** Local and incoming cases follow different paths.
- *Sample:* 437 of 772 Legal Amazon municipalities have a garimpo case in 2004–2024.
- *Local cases:* 9,724 in 2004, down to 2,372–2,936 in 2015–18, and back to 10,315 in
  2024.
- *Incoming cases:* 6,577 in 2004, down to 1,262 in 2015, up to 15,569 in 2022 (the
  Roraima rush), then 4,293 in 2024.

## 1. The shocks

In [2]:
dry = P.drought_years(frame)
dry.to_csv(OUT / "drought_years.csv")
display(dry.round(2).T)
municipalities = sample.drop_duplicates("mun_id")
upstream = sample[sample["year"].between(*P.C11_YEARS)]
display(pd.DataFrame({
    "municipalities with upstream garimpo (200 km)": upstream.groupby("year")["upstream_garimpo_200"].apply(
        lambda d: int((d > 0).sum())),
    "mean upstream garimpo, pp (200 km)": upstream.groupby("year")["upstream_garimpo_200"].mean(),
    "within-municipality SD (200 km)": upstream.groupby("year")["upstream_garimpo_200"].std(),
}).iloc[::3].round(4).T)
within = upstream["upstream_garimpo_200"] - upstream.groupby("mun_id")["upstream_garimpo_200"].transform("mean")
print("within-municipality SD of upstream garimpo, 2004-2019:", round(within.std(), 4), "pp")
display(sample[["drought", "drought_lag", "drought_upstream", "upstream_garimpo_200", "c11_garimpo_pp"]].corr().round(2))

year,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
mean,-0.21,0.11,-0.32,0.39,-0.72,-1.22,0.5,-0.79,0.49,-0.49,-0.31,0.9,0.25,-0.18,-0.62,-0.08,0.18,-0.33,0,1.29,1.17
share_dry,0.03,0.06,0.01,0.07,0,0.03,0.29,0,0.32,0.08,0.02,0.62,0.25,0,0,0.14,0.22,0,0.17,0.64,0.6


year,2004,2007,2010,2013,2016,2019
municipalities with upstream garimpo (200 km),182,182,182,183,183,183
"mean upstream garimpo, pp (200 km)",0.0478,0.0494,0.0518,0.0562,0.0625,0.0698
within-municipality SD (200 km),0.1757,0.1788,0.1848,0.2005,0.2286,0.2579


within-municipality SD of upstream garimpo, 2004-2019: 0.0481 pp


,drought,drought_lag,drought_upstream,upstream_garimpo_200,c11_garimpo_pp
drought,1,0.21,0.95,0.01,0.02
drought_lag,0.21,1,0.19,0.01,0.02
drought_upstream,0.95,0.19,1,0.01,0.02
upstream_garimpo_200,0.01,0.01,0.01,1,0.58
c11_garimpo_pp,0.02,0.02,0.02,0.58,1


**Reading the results.** Drought varies by region and year; upstream garimpo barely varies
within municipalities.
- *Drought years:* the share of garimpo municipalities with a precipitation deficit above
  1 SD is 29% in 2010, 32% in 2012, 62% in 2015, 25% in 2016, 64% in 2023 and 60% in
  2024, and under 10% in most other years. Local and upstream precipitation deficits
  correlate at 0.95, so the upstream variant adds little.
- *Upstream garimpo:*
  - 183 of the 437 municipalities have C11 garimpo within 200 km upstream. The mean share
    rises from 0.048 pp (2004) to 0.070 pp (2019).
  - Within municipalities its SD is only 0.048 pp, and it correlates 0.58 with the
    municipality's own garimpo share.

## 2. Who responds to what

In [3]:
table = P.run_table(frame)
table.to_csv(OUT / "responses.csv", index=False)
for value in ("estimate", "p_basin", "p_wcb"):
    print(value)
    display(table.pivot_table(index=["spec", "term"], columns="outcome", values=value, sort=False)
            .reindex(columns=list(P.OUTCOMES)).round(3))
table[["spec", "n_obs", "n_municipalities", "n_basins"]].drop_duplicates("spec")

estimate


outcome                                                                       log1p_garimpo_local  log1p_garimpo_incoming  local_minus_incoming  log1p_other_malaria
spec                                   term                                                                                                                         
drought                                drought                                              0.033                   0.031                 0.002                0.029
                                       drought_lag                                          0.007                   0.026                -0.019                0.029
drought, state x year FE               drought                                              0.006                   0.007                -0.001               -0.002
                                       drought_lag                                         -0.007                   0.008                -0.015                -0.02
drought, without Roraima               drought                                              0.032                   0.036                -0.004                0.033
                                       drought_lag                                          0.014                   0.034                 -0.02                0.027
drought upstream                       drought_upstream                                     0.031                   0.032                -0.001                0.013
                                       drought_upstream_lag                                 0.003                   0.021                -0.017                0.027
upstream garimpo 200 km                upstream_garimpo_200                                -0.341                  -0.247                -0.093               -1.231
upstream garimpo 50 km                 upstream_garimpo_50                                 -0.185                  -0.138                -0.047               -1.049
gold price x deposits                  gold_unexploited_any__x__log_gold_brl                0.089                    0.29                  -0.2                0.211
gold price x deposits, without Roraima gold_unexploited_any__x__log_gold_brl               -0.018                   0.106                -0.123                0.102
gold price x deposits, 2004-2018       gold_unexploited_any__x__log_gold_brl               -0.198                   0.002                  -0.2                0.068
enforcement presence                   presence__x__post2019                                 0.08                   0.098                -0.018                0.057
                                       presence__x__post2023                               -0.032                  -0.016                -0.016               -0.065

p_basin


outcome                                                                       log1p_garimpo_local  log1p_garimpo_incoming  local_minus_incoming  log1p_other_malaria
spec                                   term                                                                                                                         
drought                                drought                                              0.072                   0.094                  0.83                0.419
                                       drought_lag                                          0.684                   0.182                  0.02                0.436
drought, state x year FE               drought                                               0.75                   0.687                 0.965                 0.97
                                       drought_lag                                          0.689                   0.628                 0.207                0.674
drought, without Roraima               drought                                              0.091                   0.053                 0.633                0.374
                                       drought_lag                                           0.44                   0.077                 0.018                0.476
drought upstream                       drought_upstream                                     0.158                   0.197                 0.935                0.765
                                       drought_upstream_lag                                 0.872                   0.366                 0.062                0.503
upstream garimpo 200 km                upstream_garimpo_200                                 0.266                   0.446                 0.258                0.022
upstream garimpo 50 km                 upstream_garimpo_50                                  0.541                    0.62                 0.684                0.017
gold price x deposits                  gold_unexploited_any__x__log_gold_brl                0.523                   0.137                 0.021                0.254
gold price x deposits, without Roraima gold_unexploited_any__x__log_gold_brl                0.869                    0.34                 0.042                0.563
gold price x deposits, 2004-2018       gold_unexploited_any__x__log_gold_brl                 0.36                   0.993                 0.047                0.806
enforcement presence                   presence__x__post2019                                0.103                   0.017                 0.528                0.264
                                       presence__x__post2023                                0.382                   0.539                 0.502                0.137

p_wcb


outcome                                                                       log1p_garimpo_local  log1p_garimpo_incoming  local_minus_incoming  log1p_other_malaria
spec                                   term                                                                                                                         
drought                                drought                                              0.105                    0.13                 0.838                 0.43
                                       drought_lag                                          0.695                   0.238                 0.028                0.445
drought, state x year FE               drought                                              0.752                   0.692                 0.967                0.971
                                       drought_lag                                            0.7                   0.653                 0.205                0.679
drought, without Roraima               drought                                              0.136                   0.085                 0.642                0.392
                                       drought_lag                                          0.467                   0.117                 0.024                0.491
drought upstream                       drought_upstream                                     0.223                   0.264                 0.947                0.772
                                       drought_upstream_lag                                 0.878                   0.424                 0.079                0.508
upstream garimpo 200 km                upstream_garimpo_200                                 0.238                   0.399                 0.674                0.453
upstream garimpo 50 km                 upstream_garimpo_50                                  0.499                   0.607                 0.746                 0.19
gold price x deposits                  gold_unexploited_any__x__log_gold_brl                0.584                   0.184                 0.023                0.275
gold price x deposits, without Roraima gold_unexploited_any__x__log_gold_brl                0.878                   0.422                  0.06                0.568
gold price x deposits, 2004-2018       gold_unexploited_any__x__log_gold_brl                0.401                   0.996                 0.064                0.808
enforcement presence                   presence__x__post2019                                0.107                   0.016                 0.543                0.296
                                       presence__x__post2023                                0.516                    0.56                 0.539                0.142

,spec,n_obs,n_municipalities,n_basins
0,drought,8740,437,112
8,"drought, state x year FE",8740,437,112
16,"drought, without Roraima",8440,422,110
24,drought upstream,8620,431,111
32,upstream garimpo 200 km,6992,437,112
36,upstream garimpo 50 km,6992,437,112
40,gold price x deposits,9177,437,112
44,"gold price x deposits, without Roraima",8862,422,110
48,"gold price x deposits, 2004-2018",6555,437,112
52,enforcement presence,4370,437,112


**Reading the results.** The residents' side shows nothing yet. The one group difference,
for the price × deposits term, is a shift in the composition of workers in deposit
municipalities, and is hard to tell from a trend.
- *Drought this year:*
  - Local and incoming cases both rise about 0.03 per SD (p 0.05–0.09), as does other
    malaria (0.03, ns). That reads as transmission, and the difference is zero.
  - With state × year FE the drought terms vanish for both groups and for other malaria
    (|β| ≤ 0.02, p ≥ 0.6).
- *Drought last year:*
  - Incoming cases rise more than local ones: the difference is −0.019 (p 0.02, bootstrap
    0.03), and −0.020 without Roraima.
  - This is the opposite of the prediction that residents turn to mining after drought,
    and it is not robust to state × year FE (−0.015, p 0.21).
- *Upstream garimpo:* no response in either group (−0.25 to −0.34 per pp at 200 km,
  p ≥ 0.27; the difference p 0.26). Non-garimpo malaria falls (−1.2, Otto-3 p 0.02,
  bootstrap 0.45). With so little variation within municipalities, the test has little
  power.
- *Gold price × deposits:*
  - Local minus incoming is −0.20 (p 0.02); −0.12 without Roraima (p 0.04, bootstrap
    0.06); and −0.20 in 2004–2018 (p 0.047), before the 2019 break.
  - Before 2019 this comes from local cases falling (−0.20, p 0.36), not incoming cases
    rising (0.00). The local share of garimpo cases fell in deposit municipalities while
    the price rose.
  - The price in reais moves almost like a trend, so this cannot be separated from a
    deposit-specific trend in the composition of workers.
- *Enforcement presence × post-2019:* both groups respond (incoming +0.098, p 0.017;
  local +0.080, p 0.10). The difference (−0.018) is not significant.